# Обработка данных с помощью Polars

**Общая цель:** приобретение базовых навыков загрузки, проверки, очистки, преобразования и агрегирования данных с помощью Polars, а также получение представления об основных концепциях API и о том, как эти операции вписываются в типичный процесс анализа данных.

## Установка

Для начала импортируем библиотеку Polars. Стандартный псевдоним — `pl`.

In [78]:
import polars as pl
import polars.selectors as cs # Для использования селекторов столбцов, таких как cs.numeric()

#  Выводим версию Polars для справки
print(f"Polars version: {pl.__version__}")

Polars version: 1.35.2


Данные: https://disk.yandex.ru/d/fJQ6wvqPYwYgqA

---
## 1: Введение в обработку данных и Polars

В этом разделе представлены основные концепции обработки данных  и библиотека Polars — мощный инструмент для этих целей.  Эффективная обработка данных — важнейший этап для качественного бизнес-анализа и принятия решений на основе данных.

### 1.1. Необходимость обработки данных в бизнес-аналитике

В контексте бизнес-аналитики под **обработкой данных** понимается процесс преобразования и сопоставления необработанных данных из их исходного состояния в чистый, структурированный и проверенный формат, пригодный для анализа. Этот процесс также часто называют «очисткой данных».

* **Правило «80/20» при подготовке данных:** в сообществе специалистов по анализу данных широко распространено мнение, что подготовка данных, в том числе их обработка, может занимать до 80 % общего времени и усилий, затрачиваемых на аналитический проект. Оставшиеся 20 % обычно уходят на сам анализ и получение результатов. Это говорит о том, что на обеспечение качества и удобства использования данных тратится много ресурсов.
* **Источники данных и необходимость их повторного исполльзования:**
    Данные, используемые для бизнес-аналитики, часто поступают из систем, изначально не предназначенных для решения конкретных аналитических задач. К таким источникам относятся:
    * **Системы оперативной обработки транзакций (Online Transaction Processing,OLTP):** операционные системы (например, системы управления продажами, CRM, ERP), оптимизированные для регистрации транзакций, а не для выполнения сложных запросов.
    * **Системы оперативной аналитической обработки (Online Analytical Processing, OLAP):** предназначены для анализа, но для решения конкретных задач может потребоваться дополнительная обработка или интеграция данных.
    * **Озера данных:** репозитории с огромным объемом необработанных данных, требующих серьезной обработки. Озеро данных (Data Lake) — это централизованное хранилище, которое принимает и сохраняет большие массивы необработанных данных в их исходном, «сыром» формате. Ключевая идея в том, что структура и преобразования применяются не на этапе загрузки, а в момент, когда данные начинают использовать для анализа, машинного обучения или других задач. Этот подход называется схемой при чтении (schema-on-read), в отличие от традиционных хранилищ данных, где используется схема при записи (schema-on-write).
    * **Системы управления и архивирования нормативных данных:** данные структурированы в соответствии с требованиями, но часто требуют преобразования для анализа.

    Поскольку эти различные системы выполняют разные основные функции, получаемые ими данные редко полностью соответствуют схеме (структуре), необходимой для конкретного аналитического исследования.  Таким образом, обработка данных становится важным промежуточным этапом.

* **Типичные проблемы с данными** (к примеру, наши наборы данных `customers.csv` и `orders.csv`):
    * Несовпадающие форматы дат.
    * Пропущенные значения в критически важных полях.
    * Выбросы или ошибочные записи данных.

### 1.2. Знакомство с Polars: высокопроизводительной библиотекой для работы с датафреймами

Polars — это современная библиотека для работы с датафреймами, написанная на Rust, с высокой производительностью и выразительным API на Python.
* **Преимущества:** скорость, выразительный синтаксис, современные возможности.

### 1.3. Основные концепции Polars: обзор на высоком уровне

* **Выражения:** шаблоны для операций (`pl.col()`, `pl.lit()`).
* **Стратегии выполнения (ленивая и  нетерпеливая):** Polars может оптимизировать план операций (ленивая стратегия/обработка) или выполнять их немедленно (нетерпеливая стратегия/обработка, как в Pandas).

### 1.4. Настройка: импорт библиотеки Polars

Мы уже сделали это в ячейке настройки выше: `import polars as pl`.

---
## 2: Получение данных в режиме ввода-вывода данных

В этом разделе рассмотрена загрузка данных во фреймы данных Polars из распространенных форматов файлов.

**Примечание для пользователей Google Colab:** вы можете использовать альтернативную загрузку файлов `customers.csv` и `orders.csv` в свою среду Colab.  Для этого нажмите на значок папки на левой боковой панели, а затем на значок "Загрузить в хранилище сеанса".

In [ ]:
## TODO: если вы используете Google Colab, раскомментируйте и выполните эти строки, чтобы загрузить файлы.
from google.colab import files
print("Please upload customers.csv and orders.csv")
uploaded = files.upload()
#
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Please upload customers.csv and orders.csv


### 2.1 Чтение файлов с разделителями-запятыми (CSV)

Мы используем `pl.read_csv()` для загрузки данных из CSV-файлов.

In [ ]:
# Загрузим набор данных customers
customers_df = pl.read_csv('customers.csv')
print("Файл customers.csv успешно загружен.")
# Выведем часть для проверки
print("Фрагмент файла customers_df:")
customers_df.head(3)

In [ ]:
# Загрузим набор данных ordess
orders_df = pl.read_csv('orders.csv')
print("Файл orders.csv успешно загружен.")
# Выведем часть для проверки
print("Фрагмент файла orders_df:")
orders_df.head(3)


#### Указание или переопределение типов данных (схемы) с помощью `dtypes`

При импорте можно явно указать типы данных столбцов для обеспечения корректности, повышения производительности или эффективности использования памяти.

In [ ]:
# Определим желаемую схему для выбранных столбцов в customers.csv
customer_schema = {
    'customer_id': pl.Int32,
    'name': pl.Utf8,
    'registration_date': pl.Utf8, # Чтение в виде строки; разбор будет далее
    'city': pl.Categorical,
    'age': pl.Int64 # Разрешить значения null, но пока оставить как Int64
}

In [ ]:
customers_custom_schema_df = pl.read_csv('customers.csv', schema_overrides=customer_schema)
print("Файл customers.csv успешно загружен с пользовательской схемой.")
print("Схема файла customers_custom_schema_df:")
customers_custom_schema_df.schema

В последующих разделах мы в основном будем использовать изначально загруженные таблицы customers_df и orders_df (без пользовательской схемы для customers_df), чтобы продемонстрировать такие преобразования, как приведение типов.  Однако давайте переименуем исходные столбцы с датами, чтобы было понятно, что это строки, прежде чем разбирать их в последующих разделах.

In [ ]:
customers_df = customers_df.rename({"registration_date":"registration_date_str"})
orders_df = orders_df.rename({"order_date": "order_date_str"})

### 2.2 Чтение данных CSV с URL

Пример кода из [Руководства пользователя Polars](https://docs.pola.rs/user-guide/expressions/window-functions/)

In [ ]:
types = (
    "Grass Water Fire Normal Ground Electric Psychic Fighting Bug Steel "
    "Flying Dragon Dark Ghost Poison Rock Ice Fairy".split()
)
# Создадим тип перечисления
type_enum = pl.Enum(types)
# теперь давайте загрузим данные в формате CSV с информацией о покемонах
pokemon = pl.read_csv(
    "https://gist.githubusercontent.com/ritchie46/cac6b337ea52281aa23c049250a4ff03/raw/89a957ff3919d90e6ef2d34235e6bf22304f3366/pokemon.csv",
).cast({"Type 1": type_enum, "Type 2": type_enum})
print(pokemon.head())


### 2.3. Чтение файлов JSON

Polars может читать файлы JSON, которые обычно представляют собой список записей.

Для начала создадим в нашем рабочем каталоге образец JSON-файла с именем `product_inventory.json`. Это можно сделать, выполнив приведенный ниже код, который записывает строку JSON в файл.

In [ ]:
product_inventory_json_content = """
[
  {\"product_id\": \"P1001\", \"product_name\": \"Laptop X1\", \"category\": \"Electronics\", \"stock_level\": 150, \"reorder_point\": 50},
  {\"product_id\": \"P1002\", \"product_name\": \"Wireless Mouse\", \"category\": \"Electronics\", \"stock_level\": 300, \"reorder_point\": 100},
  {\"product_id\": \"P1003\", \"product_name\": \"Office Chair Pro\", \"category\": \"Furniture\", \"stock_level\": 85, \"reorder_point\": 30},
  {\"product_id\": \"P1004\", \"product_name\": \"Standing Desk\", \"category\": \"Furniture\", \"stock_level\": 60, \"reorder_point\": 20}
]
"""

with open('product_inventory.json', 'w') as f:
    f.write(product_inventory_json_content)

print("'product_inventory.json' успешно создан.")

In [ ]:
inventory_df = pl.read_json('product_inventory.json')
print("\nДанные о товарных запасах успешно загружены from JSON.")
inventory_df.head()

---
## 3: Первый взгляд — первичное изучение и понимание данных

В этом разделе рассматриваются основные методы проверки DataFrames.

**Прежде чем продолжить, убедитесь, что `customers_df` и `orders_df` загружены из раздела 2.**


### 3.1. Понимание структуры DataFrame: измерения (`.shape`)

In [ ]:
customer_dimensions = customers_df.shape
print(f"Датафрейм клиентов - строки: {customer_dimensions[0]}, столбцы: {customer_dimensions[1]}")

order_dimensions = orders_df.shape
print(f"Датафрейм заказов - строки: {order_dimensions[0]}, столбцы: {order_dimensions[1]}")

### 3.2. Проверка содержимого данных: первая и последняя строки (`.head()`, `.tail()`)



In [ ]:
print("Первые три строки файла customers_df:")
customers_df.head(3)

In [ ]:
# Выведите последние 4 строки файла orders_df
print("\nLast 4 rows of orders_df:")
# TODO: Напишите здесь свой код

### 3.3.  Изучение типов данных и схемы  (`.dtypes`, `.schema`)

In [ ]:
print("Типы данных для customers_df:")
print(customers_df.dtypes)

print("\nСхема для customers_df:")
print(customers_df.schema)


### 3.4. Получение описательных статистик и краткий обзор (`.describe()`, `.glimpse()`)

In [ ]:
print("Описательная статистика для orders_df:")
orders_df.describe() # Обычно отображаются числовые столбцы

In [ ]:
print("Краткий обзор customers_df:")
customers_df.glimpse()

### 3.5. Выявление и количественная оценка пропущенных значений с помощью выражений

In [ ]:
customers_df.select(
    pl.all().is_null().sum()
)

In [ ]:
# Подсчитайте количество пропущенных значений в каждом столбце orders_df, используя API выражений
print("\nКоличество пропущенных значений по столбцам в orders_df (через выражения):")
# TODO: Напишите здесь свой код


### 3.6. Анализ уникальных значений в столбцах и создание перечислений с использованием выражений

In [ ]:
print("Количество уникальных городов в customers_df (через выражение):")
customers_df.select(
    pl.col('city').n_unique().alias("unique_city_count")
)

In [ ]:
print("\nУникальные города в customers_df (через выражение):")
customers_df.select(
    pl.col('city').unique().sort()
)

---
## 4: Основные понятия Polars — строительные блоки

В этом разделе рассматриваются фреймы данных, ряды данных и выражения.

### 4.1.  Пересмотр фрейма данных Polars
Двумерная табличная структура данных в памяти.

In [ ]:
data_dict = {
    'StudentID': [1001, 1002, 1003],
    'Course': ['Finance', 'Marketing', 'Finance'],
    'MidtermScore': [85, 92, 78]
}
scores_df = pl.DataFrame(data_dict)
print("Фрейм данных, созданный с нуля:")
scores_df

### 4.2. Polars Series


Одномерный объект, похожий на массив, представляющий собой один столбец.

In [ ]:
city_series = customers_df.get_column('city')
print("Столбец 'city' извлечен в виде Series:")
city_series.head()

In [ ]:
print(f"Тип city_series: {type(city_series)}")
print(f"Название city_series: {city_series.name}")
print(f"Тип даннах city_series: {city_series.dtype}")

### 4.3. Выражения Polars: Базовый механизм
Шаблоны для операций, оцениваемых в контексте.

#### Обращение к столбцам: `pl.col()`
#### Литеральные значения: `pl.lit()`
#### Основные операции с выражениями

In [ ]:
original_price_expr = pl.col("unit_price")
discount_rate_lit_expr = pl.lit(0.05) # 5% discount
discounted_price_expr = original_price_expr * (pl.lit(1) - discount_rate_lit_expr)
print(f"Выражение для цены со скидкой: {discounted_price_expr}")

is_older_than_30_expr = pl.col("age") > pl.lit(30)
print(f"Выражение для 'age > 30': {is_older_than_30_expr}")

#### Выражения выполняются в контекстах (подробнее в разделах 5 и 6)
 В разных контекстах выражения используются для разных целей.

In [ ]:
print("Выражение в контексте `select`")
customer_status_df = customers_df.select([
    pl.col("name"),
    pl.col("age"),
    (pl.col("age") < pl.lit(18)).alias("is_minor")
])
customer_status_df.head()

In [ ]:
print("\nВыражение в контексте with_columns")
orders_with_total = orders_df.with_columns(
    (pl.col("quantity") * pl.col("unit_price")).alias("total_price")
)
orders_with_total.head()

In [ ]:
print("Выражение в контексте `filter`")
# В этом примере используется необработанный столбец «age». Обратите внимание на выброс.
adult_customers_df = customers_df.filter(
    pl.col("age") > pl.lit(30)
)
adult_customers_df.select(["name", "age"])


In [ ]:
print("Выражение в контексте `group_by().agg()`")
sales_by_category = orders_df.group_by("product_category").agg([
    pl.sum("quantity").alias("total_quantity_sold"),
    pl.mean("unit_price").alias("average_unit_price")
])
sales_by_category

---
## 5: Преобразование данных — выборка, фильтрация и изменение

В этом разделе рассматриваются основные операции по преобразованию данных.

**Убедитесь, что `customers_df` и `orders_df` загружены, а имена столбцов `registration_date_str` и `order_date_str` заданы в соответствии с изменениями, описанными в разделе 2.**

### 5.1. Выбор и изменение столбцов с помощью `select`




In [ ]:
customer_locations_df = customers_df.select([
    pl.col("name"),
    pl.col("city")
])
print("Выбранные имена и города клиентов:")
print(customer_locations_df.head())

renamed_customers_df = customers_df.select([
    pl.col("customer_id").alias("ID"),
    pl.col("name").alias("Customer Name")
])
print("\nДатафрейм клиентов с переименованными столбцамиs:")
print(renamed_customers_df.head())

#### Использование селекторов Polars (`cs`)

In [ ]:
string_cols_orders_df = orders_df.select(cs.string())
print("Строковые столбцы из orders_df:")
print(string_cols_orders_df.head())

# Выберите все числовые столбцы из orders_df
print("\nЧисловые столбцы из orders_df:")
# TODO: Напишите здесь свой код
numeric_cols_orders_df = orders_df.select(cs.numeric())
print(numeric_cols_orders_df.head())

### 5.2. Добавление или изменение столбцов с `with_columns`

In [ ]:
orders_enhanced_df = orders_df.with_columns([
    (pl.col("quantity") * pl.col("unit_price")).alias("total_price"),
    pl.col("discount_applied").is_not_null().alias("has_discount")
])
print("Датафрейм заказов со стобцами 'total_price' и 'has_discount':")
print(orders_enhanced_df.head())

In [ ]:
# В таблице customers_df создайте новый столбец name_length, в котором будет указана длина имени клиента.
customers_with_name_length_df = customers_df.with_columns(
    # TODO: Напишите здесь ваше выражение
    pl.col("name").str.len_chars().alias("name_length")
)
print("\nДатафрейм клиентов с name_length:")
print(customers_with_name_length_df.select(["name", "name_length"]).head())


### 5.3. Filtering Rows with `filter`

In [ ]:
electronics_orders_df = orders_df.filter(
    pl.col("product_category") == pl.lit("Electronics")
)
print("Электронные заказы:")
print(electronics_orders_df.head())

In [ ]:
# Примечание: в столбце 'age' в файле customers.csv есть выбросы (3000) и пропущенные значения.
# Для реалистичной фильтрации это необходимо учитывать.
# Давайте создадим обработанную (очищенную) версию для этого примера фильтрации.
customers_cleaned_age_df = customers_df.with_columns(
    pl.when(pl.col("age") > 100).then(None).otherwise(pl.col("age")).cast(pl.Int64, strict=False).alias("age_cleaned")
)
older_customers_df = customers_cleaned_age_df.filter(
    pl.col("age_cleaned") > pl.lit(50)
)
print("\nКлиенты старше 50 лет (возраст очищен):")
print(older_customers_df.select(["name", "age_cleaned", "city"]))

In [ ]:
# Отфильтруйте orders_df по товарам из категории «Home Goods» или по товарам, количество которых превышает 2.
print("\nHome Goods orders OR quantity > 2:")
# TODO: Напишите здесь свой код
filtered_orders_complex = orders_df.filter(
    (pl.col("product_category") == pl.lit("Home Goods")) | (pl.col("quantity") > pl.lit(2))
)
print(filtered_orders_complex)

### 5.4. Преобразование типов данных (кастинг)

Использование `pl.col().cast(DataType)` или специальных методов преобразования, таких как `str.to_datetime()`.

In [ ]:
# Преобразование 'city' в категориальный признак в customers_df
customers_city_categorical_df = customers_df.with_columns(
    pl.col("city").cast(pl.Categorical).alias("city_categorical")
)
print("Датафрейм клиентов с преобразованием столбца city в категориальные данные:")
print(customers_city_categorical_df.select(["name", "city_categorical"]).head())
print(f"Тип данных 'city_categorical': {customers_city_categorical_df.get_column('city_categorical').dtype}")

In [ ]:
# Преобразование 'order_date_str' в orders_df в тип данных Datetime
orders_parsed_dates_df = orders_df.with_columns(
    pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime")
)
print("\nДатафрейм заказов с преобразованием столбца order_datetime:")
print(orders_parsed_dates_df.select(["order_id", "order_datetime"]).head())
print(f"Тип данных 'order_datetime': {orders_parsed_dates_df.get_column('order_datetime').dtype}")

In [ ]:
# Разбор 'registration_date_str' в customers_df (обрабатывает смешанные форматы и null)
customers_parsed_reg_dates_df = customers_df.with_columns(
    pl.coalesce([
        pl.col("registration_date_str").str.to_date(format="%Y-%m-%d", strict=False),
        pl.col("registration_date_str").str.to_date(format="%m/%d/%Y", strict=False)
    ]).alias("registration_date")
)
print("\nCustomers DataFrame with parsed registration_date:")
print(customers_parsed_reg_dates_df.select(["name", "registration_date_str", "registration_date"]))
print(f"Data type of 'registration_date': {customers_parsed_reg_dates_df.get_column('registration_date').dtype}")


### 5.5. Обработка пропущенных значений
Использование `fill_null()` или `drop_nulls()`.

In [ ]:
customers_cleaned_age_df = customers_df.with_columns(
  pl.when(pl.col("age") > 100).then(None).otherwise(pl.col("age")).cast(pl.Float64, strict=False).alias("age_cleaned")
)

# Рассчитайте средний возраст для заполнения, исключив экстремальные выбросы для получения более разумного среднего значения
# Примечание: .item() извлекает одно значение из DataFrame/Series размером 1x1
mean_age_val = customers_cleaned_age_df.select(pl.col("age_cleaned").mean()).item()

customers_filled_age_df = customers_cleaned_age_df.with_columns(
    pl.col("age_cleaned").fill_null(mean_age_val).alias("age_filled_with_mean")
)
print(f"Даатафрейм клиентов с пропущенными значениями 'age_cleaned' заполняется средними значениями ({mean_age_val:.2f}):")
print(customers_filled_age_df.filter(customers_df["age"].is_null()).select(["name", "age", "age_cleaned", "age_filled_with_mean"]))

In [ ]:
# Удаление строк с пустыми значениями
# Удалить строки из orders_df, в которых значение 'quantity' равно null (для примера используется исходный orders_df)
orders_dropped_null_quantity_df = orders_df.drop_nulls(subset=["quantity"])
print(f"Изначальное число заказов: {orders_df.height}, После удаления строк, где не указано кочичество заказовy: {orders_dropped_null_quantity_df.height}")

### 5.6. Сортировка данных с помощью `sort()`

In [ ]:
sorted_customers_by_age_df = customers_cleaned_age_df.sort("age_cleaned", descending=True)
print("Клиенты отсортированы по возрасту (по убыванию):")
print(sorted_customers_by_age_df.select(["name", "age_cleaned", "city"]).head())

---
## 6: Агрегирование и изменение формы данных — обобщение для получения дополнительной информации

В этом разделе рассказывается о группировке данных и применении функций агрегирования.

### 6.1. Группировка и агрегация (`.group_by().agg()`)
Стратегия «разделяй-применяй-объединяй».

In [ ]:
category_summary_df = orders_df.group_by("product_category").agg([
    pl.sum("quantity").alias("total_quantity_sold"),
    pl.mean("unit_price").alias("average_unit_price"),
    pl.col("order_id").count().alias("number_of_orders"),
    pl.n_unique("customer_id").alias("unique_customers_in_category")
]).sort("total_quantity_sold", descending=True)

print("Сводка продаж по категориям товаров:")
print(category_summary_df)

In [ ]:
# Используя customers_cleaned_age_df, сгруппируйте данные по 'city' и найдите:
# 1. Количество клиентов в каждом городе.
# 2. Средний "очищенный" возраст клиентов в каждом городе.
# Отсортируйте результаты по количеству клиентов в порядке убывания.
print("\nДемографические данные клиентов по городам:")
# TODO: Напишите свой код здесь

### 6.2. Поворот данных с помощью `df.pivot()`

In [ ]:
# Убедимся, что order_datetime был проанализирован для извлечения года
if 'order_datetime' not in orders_df.columns:
    current_orders_df = orders_df.with_columns(
          pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime")
    )
else:
    current_orders_df = orders_df

orders_with_year_df = current_orders_df.with_columns(
    pl.col("order_datetime").dt.year().alias("order_year")
)

yearly_category_sales = orders_with_year_df.group_by(["order_year", "product_category"]).agg(
    pl.sum("quantity").alias("total_quantity")
).sort(["order_year", "product_category"])

print("Годовые продажи по категориям (длинный формат):")
print(yearly_category_sales.head())

pivoted_yearly_sales_df = yearly_category_sales.pivot(
    values="total_quantity",
    index="order_year",
    columns="product_category"
).fill_null(0)

print("\nРазворот годовых продаж (категории товоров в виде столбцов):")
print(pivoted_yearly_sales_df)

### 6.3. Оконные функции
Оконные функции выполняют вычисления для набора строк таблицы, связанных с текущей строкой. Примеры: нарастающие итоги (или кумулятивная сумма) — это последовательное сложение значений, где каждый следующий результат включает в себя все предыдущие); скользящие средние и ранжирование внутри групп.

In [ ]:
# Убедитесь, что order_datetime проанализировано, а quantity имеет числовое значение
if 'order_datetime' not in orders_df.columns:
    current_orders_df_window = orders_df.with_columns([
          pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime"),
          pl.col("quantity").fill_null(0) # Убедитесь, что количество заполнено для cumsum
    ])
else:
    current_orders_df_window = orders_df.with_columns(pl.col("quantity").fill_null(0))

orders_sorted_for_window = current_orders_df_window.sort(["product_category", "order_datetime"])

orders_with_running_total = orders_sorted_for_window.with_columns(
    pl.col("quantity").cum_sum().over("product_category").alias("running_total_quantity_in_category")
)
print("Заказы снарастающим итогом по категориям (иллюстративно):")
print(orders_with_running_total.filter(pl.col("product_category") == "Books").select([
    "order_datetime", "product_category", "quantity", "running_total_quantity_in_category"
]).tail(5))

---
## 7: Объединение и сохранение данных

В этом разделе рассказывается о соединении связанных наборов данных и сохранении обработанных данных.

### 7.1. Объединение данных: объединение датафреймов с помощью `.join()`

In [ ]:
merged_inner_df = orders_df.join(
    customers_df, # DataFrame для объединения
    on="customer_id", # Общий столбец для объединения
    how="inner"       # Тип объединения
)
print("Внутреннее объединение - заказы с реквизитами клиента (первые 5 строк):")
print(merged_inner_df.select([
    "order_id", "customer_id", "name", "city", "product_category", "unit_price"
]).head())
print(f"Форма внутреннего объединенного df: {merged_inner_df.shape}")

# Левое объединение: все клиенты с информацией об их заказах, если таковая имеется
customer_orders_left_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="left"
)
print("\nЛевое объединение - все клиенты с их заказами (если есть) (первые 10 строк):")
print(customer_orders_left_df.select([
    "customer_id", "name", "city", "order_id", "product_category"
]).sort("customer_id").head(10))
# У клиентов без заказов в поле order_id и других столбцах, относящихся к заказу, будет значение null
print("\nКлиенты, которые потенциально без заказов (order_id равно null при левом объединении):")
print(customer_orders_left_df.filter(pl.col("order_id").is_null()).select(["customer_id", "name", "city"]))

### 7.2. Сохранение DataFrames в файлы

In [ ]:
output_df_for_csv = merged_inner_df.select([
    "order_id", "customer_id", "name", "city", "product_category", "quantity", "unit_price", "registration_date_str"
]).sort("order_id") # Выберите еще несколько столбцов для полноты картины

In [ ]:
output_df_for_csv.write_csv("enriched_orders_report.csv")
print("\nФайл 'enriched_orders_report.csv' успешно сохранен")

In [ ]:
output_df_for_csv.head(10).write_json("enriched_orders_sample.json")
print("Файл  'enriched_orders_sample.json' успешно сохранен ")

### 7.3. Рабочий процесс и примечания по выполнению (кратко)

In [ ]:
# Пример цепочки методов
# Предположим, что customers_parsed_reg_dates_df и orders_parsed_dates_df были созданы в разделе 5.
# Для простоты давайте убедимся, что в наших основных датафреймах есть обработанные даты, если мы их используем.
customers_df_final = customers_df.with_columns([
    pl.coalesce([
        pl.col("registration_date_str").str.to_date(format="%Y-%m-%d", strict=False),
        pl.col("registration_date_str").str.to_date(format="%m/%d/%Y", strict=False)
    ]).alias("registration_date"),
    # Убедитесь, что age_cleaned существует
    pl.when(pl.col("age") > 100).then(None).otherwise(pl.col("age")).cast(pl.Int64, strict=False).alias("age_cleaned")
])

orders_df_final = orders_df.with_columns([
    pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime"),
    pl.col("quantity").fill_null(0), # Заполняем пустые значения количества для расчета
    pl.col("unit_price").fill_null(0.0) # Заполняем пустые значения цены для расчета
])

# Последовательная операция: найти общую сумму продаж для клиентов из Нью-Йорка по заказам «Электроника».
ny_electronics_sales = (
    orders_df_final
    .join(customers_df_final, on="customer_id", how="inner")
    .filter(
        (pl.col("city") == pl.lit("New York")) &
        (pl.col("product_category") == pl.lit("Electronics"))
    )
    .with_columns(
        (pl.col("quantity") * pl.col("unit_price")).alias("sale_amount")
    )
    .group_by("name") # Group by customer name
    .agg(
        pl.sum("sale_amount").alias("total_spent_on_electronics")
    )
    .sort("total_spent_on_electronics", descending=True)
)
print("\nChained operations - NY Electronics Sales by Customer:")
print(ny_electronics_sales)

In [ ]:
# Ленивое выполнение и .collect() снова
lazy_result_plan = (
    orders_df.lazy() # Начать ленивый запрос
    .filter(pl.col("unit_price") > pl.lit(500))
    .group_by("product_category")
    .agg(pl.sum("quantity").alias("total_high_value_quantity"))
)
print("\nLazy query plan constructed (no execution yet):")
print(lazy_result_plan)

print("\nExecuting lazy query with .collect():")
final_high_value_summary_df = lazy_result_plan.collect()
print(final_high_value_summary_df)

---
## 8. Упражнения для активного обучения

Выполните следующие задания, чтобы закрепить знания, полученные на этом семинаре. Используйте датафреймы `customers_df` и `orders_df`.

**Инструкции:**
1. Убедитесь, что файлы `customers.csv` и `orders.csv` загружены в `customers_df` и `orders_df`соответственно. Возможно, вам придется повторно запустить ячейки из раздела 2, если вы перезапустили блокнот.
 2. Не забывайте при необходимости преобразовывать типы данных (особенно для дат) и обрабатывать пропущенные значения для обеспечения точности анализа.
 3. Напишите код в Polars в ячейках, расположенных под каждым заданием.

### Задание 1. Анализ возраста клиентов

a. В `customers_df` определите и подсчитайте, сколько клиентов имеют возраст, записанный как аномальное значение (3000).
b. Создайте новый DataFrame `customers_age_cleaned_df`, в котором значения возраста старше 100 лет будут заменены на `None` (нулевое значение). Затем заполните все оставшиеся нулевые значения возраста (включая вновь созданные) медианным возрастом из *очищенного* распределения.
c. Каков средний возраст (из очищенных и заполненных данных) клиентов, зарегистрированных в 2023 году? (Подсказка: сначала нужно разобрать `registration_date_str`.)

In [ ]:
# Задание 1а: Подсчитайте выбросы по возрасту
print("Задание 1а:")
# Ваш код здесь


In [ ]:
# Задание 1b: Очистите и заполните ages
print("\nЗадание 1b:")
# Ваш код здесь



In [ ]:
# Задание 1c: Средний возраст клиентов, зарегистрированных в 2023 году
print("\nЗадание 1c:")
# Ваш код здесь

### Задание 2. Анализ заказов

a. В `orders_df` рассчитайте фактическую сумму скидки для каждого заказа (unit_price * quantity * discount_applied). Сохраните результат в новом столбце под названием `discount_value`. Обработайте случаи, когда `discount_applied` или `quantity` могут быть нулевыми (для этого расчета примите 0 для discount_applied и 1 для quantity, если они равны нулю).
b. В какой `product_category` самая высокая общая сумма `discount_value`, предоставленная покупателям?
c. Сколько уникальных клиентов было среди тех, кто сделал заказ в первом квартале (январь, февраль, март) любого года?

In [ ]:
# Задание 2a: Вычислить discount_value
print("Задание 2a:")
# Ваш код здесь

In [ ]:
# Задание 2b: Категория товаров с наибольшей общей суммой скидки
print("\nЗадание 2b:")
# Ваш код здесь

In [ ]:
# Задание 2c: Уникальные клиенты по заказам за первый квартал
print("\nЗадание 2c:")
# Ваш код здесь

### Задание 3. Комплексный анализ и результаты

а. Объедините `orders_df` (после необходимой очистки/подготовки, выполненной в рамках Задания 2, например, с добавлением `discount_value`) с `customers_df` (используя очищенные данные о возрасте из Задания 1, `customers_age_cleaned_df`).
b. В объединенном датафрейме найдите 3 города с наибольшим общим доходом от продаж (количество * цена_за_единицу_товара — размер_скидки). Для простоты, если значение `discount_value` недоступно из задания 2a, вы можете рассчитать общий доход как (количество * цена_за_единицу_товара) и не учитывать скидки в рамках этого конкретного подзадания, но имейте в виду, что это допущение.
 c. Сохраните итоговую сводную таблицу (город и общий доход от продаж) из задания 3b в файл CSV под названием `city_revenue_report.csv`.

In [ ]:
# Задание 3а: объединить очищенные данные о заказах и клиентах
print("Задание 3а:")
# Ваш код здесь

In [ ]:
# Задание 3b: Топ-3 городов по общему объему продаж
print("\nЗадание 3b:")
# Ваш код здесь

In [ ]:
# Задание 3c: Сохранить отчет о доходах города
print("\nЗадание 3c:")
# Ваш код здесь

## Polars Documentation


### User Guide

- [Expressions](https://docs.pola.rs/user-guide/expressions/)
- [Basic Operations](https://docs.pola.rs/user-guide/expressions/basic-operations/)
- [Strings](https://docs.pola.rs/user-guide/expressions/strings/)
- [Time Series](https://docs.pola.rs/user-guide/transformations/time-series/parsing/)
- [Casting](https://docs.pola.rs/user-guide/expressions/casting/)
- [Missing Data](https://docs.pola.rs/user-guide/expressions/missing-data/)
- [Enums & Categorical Data](https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/)
- [Aggregation](https://docs.pola.rs/user-guide/expressions/aggregation/)
- [Concatenation](https://docs.pola.rs/user-guide/transformations/concatenation/)
- [Join](https://docs.pola.rs/user-guide/transformations/joins/)

### API Reference

- [API Reference Home](https://docs.pola.rs/api/python/stable/reference/index.html)
- [DataFrame](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html)
- [Series](https://docs.pola.rs/api/python/stable/reference/series/index.html)
- [Expressions](https://docs.pola.rs/api/python/stable/reference/expressions/index.html )
- [LazyFrame](https://docs.pola.rs/api/python/stable/reference/lazyframe/index.html )
- [Selectors](https://docs.pola.rs/api/python/stable/reference/selectors.html )
- [IO](https://docs.pola.rs/api/python/stable/reference/io.html)

